## **Bank Nifty Predictability Under Regulatory Expiry-Structure Reform**

### **Notebook 07: Feature Ablation Study**

[![Markdown](https://img.shields.io/badge/Markdown-Ablation-blue)](https://daringfireball.net/projects/markdown/)

---

**Scope:** Run the systematic feature ablation study comparing four specific model conditions to isolate individual feature contribution.

---


### **Pipeline Context**

This is a sequential 9-notebook experimental workflow:

1. **01_data_collection.ipynb**
2. **02_data_cleaning_eda.ipynb**
3. **03_feature_engineering.ipynb**
4. **04_volatility_regime_detection.ipynb**
5. **05_baseline_models.ipynb**
6. **06_proposed_model.ipynb**
7. **07_ablation_study.ipynb**
8. **08_evaluation_and_diagnostics.ipynb**
9. **09_figures_for_paper.ipynb**

---
**Current Location:** 7. **07_ablation_study.ipynb** <- *You are here*

---


## 1. Introduction

### 1.1 Purpose of This Notebook
To prove that our engineered features, and not model architecture tuning, drive forecasting improvements, we perform a systematic feature ablation study. We keep the model architecture fixed (XGBoost Regressor) and train models across four feature configurations:
1. **Baseline**: Technical indicators only.
2. **Baseline + Expiry**: Technical + cyclical expiry features.
3. **Baseline + Regime**: Technical + HMM regime probabilities.
4. **Proposed**: Combined baseline, expiry, and regime features.

## 2. Environment Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import joblib

project_dir = Path("e:/BN_Structure_Reforms/banknifty-expiry-volatility-ml")
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src import config
from src.models import build_xgb_regressor, train_model
from src.evaluation import calculate_metrics

## 3. Run Ablation Configurations

In [2]:
train = pd.read_parquet(config.PROCESSED_DATA_DIR / "train.parquet")
val = pd.read_parquet(config.PROCESSED_DATA_DIR / "val.parquet")
test = pd.read_parquet(config.PROCESSED_DATA_DIR / "test.parquet")

# Feature lists
baseline_feats = ['RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'BB_PctB', 'BB_Width', 'ATR', 'Rolling_Vol', 'Return']
expiry_feats = ['Days_To_Expiry', 'Expiry_Week', 'Expiry_Dist_Sin', 'Expiry_Dist_Cos']
regime_feats = ['Regime_Prob_Calm', 'Regime_Prob_Stressed']

conditions = {
    "Baseline (Technical Only)": baseline_feats,
    "Baseline + Expiry": baseline_feats + expiry_feats,
    "Baseline + Regime": baseline_feats + regime_feats,
    "Proposed (Combined)": baseline_feats + expiry_feats + regime_feats
}

results = []
predictions = {}

for name, feats in conditions.items():
    X_train, y_train = train[feats], train['Target']
    X_val, y_val = val[feats], val['Target']
    X_test, y_test = test[feats], test['Target']
    
    model = build_xgb_regressor()
    model = train_model(model, X_train, y_train, X_val, y_val)
    
    preds = model.predict(X_test)
    predictions[name] = preds
    
    metrics = calculate_metrics(y_test, preds)
    metrics["Model Condition"] = name
    results.append(metrics)
    
df_results = pd.DataFrame(results)
df_results = df_results[["Model Condition", "RMSE", "MAE", "Directional_Accuracy"]]
print(df_results.to_markdown(index=False))

| Model Condition           |      RMSE |        MAE |   Directional_Accuracy |
|:--------------------------|----------:|-----------:|-----------------------:|
| Baseline (Technical Only) | 0.0122512 | 0.00879016 |               0.537415 |
| Baseline + Expiry         | 0.012331  | 0.00884272 |               0.544218 |
| Baseline + Regime         | 0.0122306 | 0.00866492 |               0.52381  |
| Proposed (Combined)       | 0.0123103 | 0.00876636 |               0.530612 |


## 4. Save Ablation Outputs

In [3]:
df_results.to_csv(config.TABLES_DIR / "ablation_results.csv", index=False)
df_results.to_markdown(config.TABLES_DIR / "ablation_results.md", index=False)
joblib.dump(predictions, config.MODELS_DIR / "ablation_predictions.joblib")
print("[OK] Saved ablation results table and predictions.")

[OK] Saved ablation results table and predictions.
